# MedPredict AI - Diabetes Model Training

This notebook teaches the complete beginner-friendly workflow for training a diabetes prediction model.

We will learn:

- What the dataset columns mean
- How to split data into train and test sets
- Why scaling is used
- How `RandomForestClassifier` works
- How to evaluate the model
- How to save `model.pkl` and `scaler.pkl`

Important: this model is educational and must not be used as a medical diagnosis system.

## 1. Dataset Structure

This notebook expects a CSV file named `diabetes.csv` in:

`app/ml/diabetes/data/diabetes.csv`

Expected columns:

| Column | Meaning |
|---|---|
| Pregnancies | Number of pregnancies |
| Glucose | Plasma glucose concentration |
| BloodPressure | Diastolic blood pressure |
| SkinThickness | Triceps skin fold thickness |
| Insulin | Serum insulin |
| BMI | Body Mass Index |
| DiabetesPedigreeFunction | Family-history based diabetes score |
| Age | Patient age |
| Outcome | Target label: 0 = negative, 1 = positive |

The model learns patterns from the feature columns to predict `Outcome`.

In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd()
if BASE_DIR.name != 'diabetes':
    BASE_DIR = Path('app/ml/diabetes')

DATASET_PATH = BASE_DIR / 'data' / 'diabetes.csv'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'

FEATURE_COLUMNS = [
    'Pregnancies',
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI',
    'DiabetesPedigreeFunction',
    'Age',
]
TARGET_COLUMN = 'Outcome'

DATASET_PATH

## 2. Load the Dataset

`pandas.read_csv()` loads the CSV into a DataFrame, which is like a spreadsheet in Python.

Each row is one patient example. Each column is one variable.

In [ ]:
df = pd.read_csv(DATASET_PATH)
df.head()

## 3. Validate the Dataset

Before training, we check that all required columns exist, no values are missing, and the target contains only `0` or `1`.

This step prevents confusing model errors later.

In [ ]:
required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing columns: {missing_columns}')

if df[required_columns].isna().any().any():
    raise ValueError('Dataset contains missing values.')

invalid_targets = set(df[TARGET_COLUMN].unique()) - {0, 1}
if invalid_targets:
    raise ValueError(f'Outcome must contain only 0 and 1. Found: {invalid_targets}')

df[required_columns].describe()

## 4. Split Features and Target

`X` contains the input features.

`y` contains the answer the model must learn to predict.

In simple terms:

- `X` = clinical measurements
- `y` = diabetes outcome label

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print('Feature shape:', X.shape)
print('Target shape:', y.shape)
print(y.value_counts(normalize=True))

## 5. Train/Test Split

We do not train and evaluate on the same rows.

Why?

If the model is evaluated on examples it already saw, the score can be misleadingly high.

We split the dataset:

- Training set: used to teach the model
- Test set: used to evaluate the model on unseen examples

`stratify=y` keeps the same proportion of positive and negative cases in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

## 6. Scaling

Scaling transforms numeric features so they have a similar magnitude.

`StandardScaler` converts each feature roughly into this form:

`scaled_value = (value - mean) / standard_deviation`

A Random Forest does not strictly require scaling, but saving a scaler is useful because the backend must transform future user inputs in exactly the same way as training data.

Important rule:

- Fit the scaler on training data only
- Use the same scaler to transform test data and future API inputs

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled[:3]

## 7. RandomForestClassifier

A Random Forest is a collection of many decision trees.

Each tree makes a prediction. The forest combines their votes.

Why it is useful for a first medical-risk model:

- Handles nonlinear patterns
- Works well on tabular data
- Less sensitive to outliers than many simple models
- Can estimate feature importance

`class_weight='balanced'` helps when the dataset has more negative cases than positive cases.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_scaled, y_train)

## 8. Model Evaluation Metrics

We use several metrics because one score is not enough.

- Accuracy: percentage of correct predictions
- Precision: among predicted positives, how many were actually positive
- Recall: among actual positives, how many did we find
- F1-score: balance between precision and recall
- ROC-AUC: how well the model separates positive and negative cases across thresholds
- Confusion matrix: counts true/false positives/negatives

For medical risk screening, recall is especially important because missing high-risk patients can be harmful.

In [ ]:
y_pred = model.predict(X_test_scaled)
y_probability = model.predict_proba(X_test_scaled)[:, 1]

evaluation = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1_score': f1_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_probability),
    'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
    'classification_report': classification_report(y_test, y_pred, zero_division=0, output_dict=True),
    'feature_columns': FEATURE_COLUMNS,
    'target_column': TARGET_COLUMN,
    'model_type': 'RandomForestClassifier',
}

json.dumps({key: evaluation[key] for key in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']}, indent=2)

In [ ]:
confusion_matrix(y_test, y_pred)

## 9. Save Model Artifacts

The backend needs two files:

- `model.pkl`: the trained Random Forest
- `scaler.pkl`: the fitted scaler

The backend must load both files. Every future prediction must pass through the same scaler before calling the model.

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, ARTIFACTS_DIR / 'model.pkl')
joblib.dump(scaler, ARTIFACTS_DIR / 'scaler.pkl')

with (ARTIFACTS_DIR / 'evaluation.json').open('w', encoding='utf-8') as file:
    json.dump(evaluation, file, indent=2)

print('Saved:', ARTIFACTS_DIR / 'model.pkl')
print('Saved:', ARTIFACTS_DIR / 'scaler.pkl')
print('Saved:', ARTIFACTS_DIR / 'evaluation.json')